# Big Data Analytics — Assignment 01
> Author : Badr TAJINI - Big Data Analytics - ESIEE 2025-2026

**Chapter 1 :** Introduction to Big Data  
**Chapter 2 :** MapReduce Algorithm Design

**Tools :** Spark or PySpark.   
**Advice:** Keep evidence and reproducibility.

## 0. Bootstrap
Use Profile A from the `BDA_Installation_Guide.md`. Log versions and key Spark configs.

In [1]:
# write some code here
# - create SparkSession('BDA-A01') with UTC timezone
# - print Spark/PySpark/Python versions
# - set spark.sql.shuffle.partitions small for local runs

import sys
import platform
from pyspark.sql import SparkSession
import pyspark

# Configuration de la Session Spark
spark = (
    SparkSession.builder
    .appName("BDA-A01")  # Nom de l'application
    .config("spark.sql.session.timeZone", "UTC")
    # Configuration recommandée pour les tests locaux (réduit les partitions de Shuffle)
    .config("spark.sql.shuffle.partitions", "8") 
    .getOrCreate()
)
sc = spark.sparkContext

# Affichage des versions
print(f"Spark Session créée | Port UI: {spark.sparkContext.uiWebUrl}")
print(f"Spark version: {spark.version}")
print(f"PySpark version: {pyspark.__version__}")
print(f"Python version: {sys.version.split()[0]} | OS: {platform.platform()}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/14 09:33:24 WARN Utils: Your hostname, Elliot, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/11/14 09:33:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/14 09:33:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session créée | Port UI: http://10.255.255.254:4040
Spark version: 4.0.1
PySpark version: 4.0.1
Python version: 3.10.19 | OS: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39


In [2]:
print(spark.sparkContext.uiWebUrl)

http://10.255.255.254:4040


## 1. Load dataset

In [3]:
## 1. Load data (Local File)
from pathlib import Path

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUTS_DIR = BASE_DIR / "outputs"
PROOF_DIR = BASE_DIR / "proof"

for directory in (DATA_DIR, OUTPUTS_DIR, PROOF_DIR):
    directory.mkdir(exist_ok=True)


FILE_NAME = "shakespeare.txt" 
TEXT_PATH = DATA_DIR / FILE_NAME

if not TEXT_PATH.exists():
    raise FileNotFoundError(f"Erreur: Le fichier {TEXT_PATH} est introuvable. Veuillez le placer manuellement dans le dossier data/.")
    
raw_rdd = spark.sparkContext.textFile(str(TEXT_PATH)).cache()

lines_df = spark.read.text(str(TEXT_PATH)).withColumnRenamed("value", "line").cache()

raw_rdd.count()
lines_df.count()

print(f"Data loaded successfully from: {TEXT_PATH}")
lines_df.show(5, truncate=False)

Data loaded successfully from: /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab1/assignment/data/shakespeare.txt
+----------------------+
|line                  |
+----------------------+
|1609                  |
|                      |
|THE SONNETS           |
|                      |
|by William Shakespeare|
+----------------------+
only showing top 5 rows


## 2. Part A — “perfect x” follower counts

In [4]:
# write some code here
# - tokenize lowercase, split on non-letters
# - for each line, if tokens[i]=='perfect' take tokens[i+1]
# - discard followers with count=1
# - write outputs/perfect_followers.csv
# - save explain('formatted') to proof/plan_perfect.txt

from pyspark.sql import functions as F
from operator import add
from contextlib import redirect_stdout
from io import StringIO
import re
import pandas as pd

token_pattern = re.compile(r"[a-z]+")

# RDD: (ligne) -> liste de mots suiveurs de 'perfect'
followers_rdd = (
    raw_rdd
    .flatMap(lambda line: [
        # 1. Tokenisation et mise en minuscule
        token_pattern.findall(line.lower())
    ])
    .flatMap(lambda tokens: [
        # 2. Extraction du follower 'x' si le mot précédent est 'perfect'
        tokens[i+1]
        for i in range(len(tokens) - 1)
        if tokens[i] == 'perfect' and tokens[i+1] 
    ])
    .map(lambda follower: (follower, 1))
    .reduceByKey(add) # Compte les occurrences
)

# Conversion en DataFrame pour le filtre final et la sauvegarde standard
perfect_counts_df = (
    followers_rdd
    .toDF(["follower", "count"])
    .filter(F.col("count") > 1)  # 3. Filtre : count > 1 (exigence de l'énoncé)
    .orderBy(F.desc("count"), F.asc("follower"))
)

#perfect_counts_df.show(10, truncate=False)

# 4. Sauvegarde du résultat (outputs/perfect_followers.csv)
perfect_counts_df.toPandas().to_csv(OUTPUTS_DIR / "perfect_followers.csv", index=False)
print(f"\nRésultats sauvegardés dans {OUTPUTS_DIR / 'perfect_followers.csv'}")

# 5. Sauvegarde du plan d'exécution (proof/plan_perfect.txt)
plan_buffer = StringIO()
with redirect_stdout(plan_buffer):
    perfect_counts_df.explain("formatted") 

(PROOF_DIR / "plan_perfect.txt").write_text(plan_buffer.getvalue())
print(f"Plan sauvegardé dans {PROOF_DIR / 'plan_perfect.txt'}")



Résultats sauvegardés dans /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab1/assignment/outputs/perfect_followers.csv
Plan sauvegardé dans /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab1/assignment/proof/plan_perfect.txt


## 3. Part B — PMI with RDDs: pairs

In [5]:
# write some code here
# - parse --threshold K
# - keep first 40 tokens per line
# - compute counts for x and (x,y); then PMI=log10(P(x,y)/(P(x)P(x)))
# - filter by threshold; write outputs/pmi_pairs_sample.csv
# - save plan text to proof/plan_pmi_pairs.txt if DF used
## 5. PMI — pairs (RDD)
import math
from itertools import combinations
from io import StringIO
from contextlib import redirect_stdout

MAX_TOKENS = 40
PMI_THRESHOLD = 5

def dedupe_preserve(tokens):
    seen = set()
    ordered = []
    for token in tokens:
        if token not in seen:
            seen.add(token)
            ordered.append(token)
    return ordered

pmi_token_pattern = re.compile(r"[a-z]+")

tokens_per_line = (
    lines_df.rdd
    .map(lambda row: [t for t in pmi_token_pattern.findall(row.line.lower())][:MAX_TOKENS])
    .map(lambda tokens: [t for t in tokens if t])
    .map(dedupe_preserve)
    .filter(lambda tokens: len(tokens) > 1)
    .cache())

num_docs = tokens_per_line.count()

from operator import add

marginal_counts = (
    tokens_per_line
    .flatMap(lambda tokens: ((token, 1) for token in tokens))
    .reduceByKey(add))

marginal_dict = dict(marginal_counts.collect())
marginal_bc = spark.sparkContext.broadcast(marginal_dict)

pair_counts = (
    tokens_per_line
    .flatMap(lambda tokens: [((min(a, b), max(a, b)), 1) for a, b in combinations(tokens, 2)])
    .reduceByKey(add)
    .filter(lambda kv: kv[1] >= PMI_THRESHOLD))

def compute_pair_pmi(kv):
    (x, y), co_count = kv
    count_x = marginal_bc.value.get(x)
    count_y = marginal_bc.value.get(y)
    if not count_x or not count_y:
        return None
    pmi = math.log10((co_count * num_docs) / (count_x * count_y))
    return (x, y, float(pmi), int(co_count))

pmi_pairs_rdd = pair_counts.map(compute_pair_pmi).filter(lambda row: row is not None)

pairs_df = spark.createDataFrame(pmi_pairs_rdd, schema=["x", "y", "pmi", "count"]).orderBy(F.desc("pmi"))

#pairs_df.show(10, truncate=False)

pairs_df.toPandas().to_csv(OUTPUTS_DIR / "pmi_pairs_sample.csv", index=False)

plan_buffer = StringIO()
with redirect_stdout(plan_buffer):
    pairs_df.explain("formatted")

(PROOF_DIR / "plan_pmi_pairs.txt").write_text(plan_buffer.getvalue())

1466

## 4. Part B — PMI with RDDs: stripes

In [6]:
# write some code here
# - build stripes x -> map[y -> count] with combiners
# - reuse univariate counts; compute PMI with log10
# - threshold K; write outputs/pmi_stripes_sample.csv
# - plan to proof/plan_pmi_stripes.txt if DF used
## 6. PMI — stripes (RDD)
from collections import Counter
from contextlib import redirect_stdout
from io import StringIO

def stripe_builder(tokens):
    for x in tokens:
        counter = Counter()
        for y in tokens:
            if y != x:
                counter[y] += 1
        if counter:
            yield (x, counter)

def merge_counters(c1, c2):
    c1.update(c2)
    return c1

def stripe_to_rows(item):
    x, counter = item
    count_x = marginal_bc.value.get(x)
    if not count_x:
        return []
    rows = []
    for y, co_count in counter.items():
        if co_count >= PMI_THRESHOLD:
            count_y = marginal_bc.value.get(y)
            if not count_y:
                continue
            pmi = math.log10((co_count * num_docs) / (count_x * count_y))
            rows.append((x, y, float(pmi), int(co_count)))
    return rows

stripes_counts = (
    tokens_per_line
    .flatMap(stripe_builder)
    .reduceByKey(merge_counters))

pmi_stripes_rdd = stripes_counts.flatMap(stripe_to_rows)

stripes_df = spark.createDataFrame(pmi_stripes_rdd, schema=["x", "y", "pmi", "count"]).orderBy(F.desc("pmi"))

stripes_df.show(10, truncate=False)

stripes_df.toPandas().to_csv(OUTPUTS_DIR / "pmi_stripes_sample.csv", index=False)

plan_buffer = StringIO()
with redirect_stdout(plan_buffer):
    stripes_df.explain("formatted")

(PROOF_DIR / "plan_pmi_stripes.txt").write_text(plan_buffer.getvalue())

+--------+--------+------------------+-----+
|x       |y       |pmi               |count|
+--------+--------+------------------+-----+
|pell    |mell    |4.267109122884396 |5    |
|mell    |pell    |4.267109122884396 |5    |
|sauf    |votre   |4.091017863828714 |5    |
|votre   |sauf    |4.091017863828714 |5    |
|margery |jourdain|4.011836617781089 |5    |
|jourdain|margery |4.011836617781089 |5    |
|med     |cine    |4.003867688109814 |7    |
|cine    |med     |4.003867688109814 |7    |
|timandra|phrynia |3.9660791272204143|5    |
|phrynia |timandra|3.9660791272204143|5    |
+--------+--------+------------------+-----+
only showing top 10 rows


1466

## 5. Spark UI evidence
Open http://localhost:4040 during runs and capture Files Read, Input Size, Shuffle Read/Write.

## 6. Environment and reproducibility

In [10]:
# write some code here
# - print Java version, Spark conf, OS info
# - save ENV.md: versions + key configs
import json
import subprocess

def get_java_version():
    try:
        output = subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT)
        return output.decode("utf-8").strip().splitlines()[0]
    except Exception as exc:
        return f"Unavailable ({exc})"

java_output = get_java_version()
print(f"Java: {java_output}")

print("Spark configuration (selected):")
conf_items = sorted(spark.sparkContext.getConf().getAll())
for key, value in conf_items:
    print(f" - {key} = {value}")

env_summary = {
    "python": sys.version,
    "spark": spark.version,
    "pyspark": pyspark.__version__,
    "java": java_output,
    "os": platform.platform(),
    "spark_conf": {k: v for k, v in conf_items if k.startswith("spark.")}}

env_lines = [
    "# Environment Summary",
    "",
    f"- Python: {sys.version.split()[0]}",
    f"- Spark: {spark.version}",
    f"- PySpark: {pyspark.__version__}",
    f"- Java: {java_output}",
    f"- OS: {platform.platform()}",
    "",
    "## Spark Configuration"]

env_lines.extend(f"- {k} = {v}" for k, v in env_summary["spark_conf"].items())

ENV_PATH = Path("ENV.md")
ENV_PATH.write_text("\n".join(env_lines) + "\n")

print(f"Environment details saved to {ENV_PATH.resolve()}")

Java: openjdk version "21.0.6" 2025-01-21
Spark configuration (selected):
 - spark.app.id = local-1761233231783
 - spark.app.name = BDA-A01
 - spark.app.startTime = 1761233229800
 - spark.app.submitTime = 1761233228747
 - spark.driver.extraJavaOptions = -Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-modules=jdk.incubator.vector --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-open